In [10]:
import os
import glob
import cv2
import seaborn as sns
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset

import warnings

warnings.filterwarnings("ignore")

In [11]:
image_gen = ImageDataGenerator()

In [12]:
train_generator = image_gen.flow_from_directory(
    'dataset/train/',
    target_size=(112, 112),
    batch_size=10,
    color_mode='grayscale',
    class_mode='binary'
)

Found 90000 images belonging to 2 classes.


In [13]:
valid_generator = image_gen.flow_from_directory(
    'dataset/valid/',
    target_size=(112, 112),
    batch_size=10,
    color_mode='grayscale',
    class_mode='binary'
)

Found 15000 images belonging to 2 classes.


In [ ]:
from keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D
from keras.layers import Dropout, Flatten, Dense
from keras.models import Sequential
from keras.layers import BatchNormalization

model = Sequential()

# Pamameters Initialization
input_shape = (224, 224, 1)
activation = 'relu'
padding = 'same'
droprate = 0.1
epsilon=0.001

model = Sequential()
model.add(BatchNormalization(input_shape=input_shape))
model.add(Conv2D(filters=16, kernel_size=3, activation=activation, padding=padding,kernel_regularizer=l2(0.001)))
model.add(MaxPooling2D(pool_size=2))
model.add(BatchNormalization(epsilon=epsilon))

model.add(Conv2D(filters=32, kernel_size=3, activation=activation, padding=padding))
model.add(MaxPooling2D(pool_size=2))
model.add(BatchNormalization(epsilon=epsilon))
model.add(Dropout(0.1))

model.add(Conv2D(filters=64, kernel_size=3, activation=activation, padding=padding))
model.add(MaxPooling2D(pool_size=2))
model.add(BatchNormalization(epsilon=epsilon))
model.add(Dropout(0.15))

model.add(Conv2D(filters=128, kernel_size=3, activation=activation, padding=padding))
model.add(MaxPooling2D(pool_size=2))
model.add(BatchNormalization(epsilon=epsilon))
model.add(Dropout(0.25))

model.add(Conv2D(filters=256, kernel_size=3, activation=activation, padding=padding))
model.add(MaxPooling2D(pool_size=2))
model.add(BatchNormalization(epsilon=epsilon))
model.add(Dropout(0.30))

model.add(Conv2D(filters=512, kernel_size=3, activation=activation, padding=padding))
model.add(MaxPooling2D(pool_size=2))
model.add(BatchNormalization(epsilon=epsilon))
model.add(Dropout(0.35))

model.add(GlobalAveragePooling2D())
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 224, 224, 1)    │             4 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 245 (980.00 B)

 Trainable params: 211 (844.00 B)

 Non-trainable params: 34 (136.00 B)

In [48]:
from tensorflow.keras.optimizers import Adam

model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])

In [49]:
history = model.fit(
    train_generator,
    steps_per_epoch = (90000//100),
    validation_data = valid_generator,
    validation_steps = (15000//100),
    epochs = 1
)

 41/900 ━━━━━━━━━━━━━━━━━━━━ 21:00 1s/step - accuracy: 0.5756 - loss: 0.6829

KeyboardInterrupt: 

In [14]:
densenet = DenseNet121( weights=None, include_top=False, input_shape=(112,112,1) )


In [15]:
model = Sequential([ 
        densenet,
        layers.GlobalAveragePooling2D(),
        layers.Dense(1, activation='sigmoid')
    ])

In [16]:
model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])


In [17]:
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 3, 3, 1024)     │     7,031,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,032,257 (26.83 MB)

 Trainable params: 6,948,609 (26.51 MB)

 Non-trainable params: 83,648 (326.75 KB)

In [19]:
history = model.fit(
    train_generator,
    steps_per_epoch = (90000//10),
    validation_data = valid_generator,
    validation_steps = (15000//10),
    epochs = 10
)

Epoch 1/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 4908s 545ms/step - accuracy: 0.7499 - loss: 0.5251 - val_accuracy: 0.7251 - val_loss: 0.5644
Epoch 2/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 4805s 534ms/step - accuracy: 0.7500 - loss: 0.5151 - val_accuracy: 0.7230 - val_loss: 0.5405
Epoch 3/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 4791s 532ms/step - accuracy: 0.7518 - loss: 0.5056 - val_accuracy: 0.7501 - val_loss: 0.6360
Epoch 4/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 4799s 533ms/step - accuracy: 0.7733 - loss: 0.4657 - val_accuracy: 0.9291 - val_loss: 0.1860
Epoch 5/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 4790s 532ms/step - accuracy: 0.9394 - loss: 0.1610 - val_accuracy: 0.9512 - val_loss: 0.1365
Epoch 6/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 4897s 540ms/step - accuracy: 0.9580 - loss: 0.1162 - val_accuracy: 0.9470 - val_loss: 0.1392
Epoch 7/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 4797s 533ms/step - accuracy: 0.9633 - loss: 0.1028 - val_accuracy: 0.9579 - val_loss: 0.1199
Epoch 8/10
9000/9000 ━━━━━━━━━━━━━━━━━━━━ 7750s 858ms/s

In [ ]:
model.save('Real_VS_Fake_densenet121.h5')

: 

In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to match CNN input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3-channel grayscale
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
train_dataset = ImageFolder(root='./dataset/train', transform=transform)
valid_dataset = ImageFolder(root='./dataset/valid', transform=transform)
test_dataset = ImageFolder(root='./dataset/test', transform=transform)

train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=100,
        num_workers=0,
        shuffle=False
    )
valid_loader = torch.utils.data.DataLoader(
        valid_dataset,
        batch_size=100,
        num_workers=0,
        shuffle=False
    )

Modeling

In [17]:
'''
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # Conv2D (32 filters)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.BatchNorm2d(32, eps=epsilon),
            nn.Dropout(droprate),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # Conv2D (64 filters)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.BatchNorm2d(64, eps=epsilon),
            nn.Dropout(droprate),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # Conv2D (128 filters)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.BatchNorm2d(128, eps=epsilon),
            nn.Dropout(droprate),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),  # Conv2D (256 filters)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.BatchNorm2d(256, eps=epsilon),
            nn.Dropout(droprate),
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1),  # Conv2D (512 filters)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.BatchNorm2d(512, eps=epsilon),
            nn.Dropout(droprate),
'''
import torch.nn as nn
import numpy as np

class MyModel(nn.Module):
    def __init__(self, input_shape=(1, 224, 224), droprate=0.1, epsilon=0.001):
        super(MyModel, self).__init__()

        self.input_shape = input_shape
        self.droprate = droprate
        self.epsilon = epsilon

        self.layer1 = nn.Sequential(
            nn.Conv2d(input_shape[0], 16, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.BatchNorm2d(16, eps=epsilon),
            nn.Dropout(droprate))
        self.fc = nn.Linear(16 * input_shape[1] * input_shape[2], 2)
    def forward(self, x):
        out = self.layer1(x)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out

input_shape = (1, 224,224)
model = MyModel(input_shape=input_shape)

print(model)

MyModel(
  (layer1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (4): Dropout(p=0.1, inplace=False)
  )
  (fc): Linear(in_features=802816, out_features=2, bias=True)
)


In [22]:
import torch.optim as optim
import numpy

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
model.to(device)

MyModel(
  (layer1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (4): Dropout(p=0.1, inplace=False)
  )
  (fc): Linear(in_features=802816, out_features=2, bias=True)
)

In [25]:
epochs = 1

for epoch in range(epochs):
    running_loss = 0
    for images, classes in train_loader:
        # Use GPU 
        images, classes = images.to(device), classes.to(device)
        
        # clears old gradients from the last step
        optimizer.zero_grad()
        
        # train the images
        outputs = model(images)
        
        #calculate the loss given the ouputs and the classes
        loss = criterion(outputs, classes)
        
        # compute the loss of every parameter
        loss.backward()
        
        # apply the optimizer and it's parameters
        optimizer.step()
        
        running_loss += loss.item()
        
    else:
        validation_loss = 0
        accuracy = 0
        
        # to make the model run faster we are using the gradients on the train
        with torch.no_grad():
            # specify that this is validation and not training
            model.eval()
            for images, classes in valid_loader:
                
                # Use GPU
                images, classes = images.to(device), classes.to(device)
                
                # validate the images
                outputs = model(images)
                
                # compute validation loss
                loss = criterion(outputs, classes)
                
                validation_loss += loss.item()
                
                # 
                ps = torch.exp(outputs)
                
                #Returns the k largest elements of the given input tensor along a given dimension.
                top_p, top_class = ps.topk(1, dim=1)
                
                # reshape the tensor
                equals = top_class == classes.view(*top_class.shape)
                
                # calculate the accuracy.
                accuracy += torch.mean(equals.type(torch.FloatTensor))
        
        # change the mode to train for other epochs
        model.train()

        print("Epoch: {}/{}.. ".format(epoch+1, epochs),
              "Training Loss: {:.3f}.. ".format(running_loss/len(train_loader)),
              "Valid Loss: {:.3f}.. ".format(validation_loss/len(valid_loader)),
              "Valid Accuracy: {:.3f}".format(accuracy/len(valid_loader)))

RuntimeError: Numpy is not available